# MovieLens 20M + Wikidata OLAP analysis

This notebook runs five multidimensional analyses against the generated SQLite warehouse. Each query is timed end-to-end: SQLite execution plus fetching all result rows. Query results are summarized in the notebook to keep the file manageable. Ratings and tags remain at their distinct fact grains; genre and country bridges are joined only for the intended analysis grain.

The queries use UTC event dates loaded from MovieLens Unix timestamps. User activity bands in Query 2 use each user's **full-dataset** rating count, while its rating measures are sliced to 2010–2015 inclusive. Query 5 reports unmatched/no-country coverage alongside country and release-decade results.

In [1]:
from pathlib import Path
import json
import platform
import sqlite3
import sys
import time

# Find the repository root whether Jupyter starts here or in notebooks/.
_root_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in _root_candidates if (p / "data/warehouse/movie_preference.sqlite").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find data/warehouse/movie_preference.sqlite from the notebook working directory")
DB_PATH = PROJECT_ROOT / "data/warehouse/movie_preference.sqlite"
OUTPUT_PATH = PROJECT_ROOT / "outputs/olap/olap_run.json"
DB_BYTES = DB_PATH.stat().st_size

con = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
con.row_factory = sqlite3.Row
con.execute("PRAGMA query_only = ON")

metadata = {
    "database": str(DB_PATH.relative_to(PROJECT_ROOT)),
    "database_bytes": DB_BYTES,
    "python_version": sys.version.split()[0],
    "sqlite_version": sqlite3.sqlite_version,
    "operating_system": platform.platform(),
    "machine_architecture": platform.machine(),
    "timing_method": "One sequential full execution per query on one read-only SQLite connection; perf_counter around execute plus fetchall. Default SQLite page cache and OS file cache; no cache flush or warmup; query order is reported.",
}
print(json.dumps(metadata, indent=2))

{
  "database": "data/warehouse/movie_preference.sqlite",
  "database_bytes": 5700124672,
  "python_version": "3.14.3",
  "sqlite_version": "3.50.4",
  "operating_system": "macOS-26.5.1-arm64-arm-64bit-Mach-O",
  "machine_architecture": "arm64",
  "timing_method": "One sequential full execution per query on one read-only SQLite connection; perf_counter around execute plus fetchall. Default SQLite page cache and OS file cache; no cache flush or warmup; query order is reported."
}


## Query 1 — Rating roll-up by year/month and MovieLens genre

Returns month-level detail (drill-down) and year-level roll-up in one result. Each rating event contributes once to each MovieLens genre assigned to its movie; Wikidata genres are excluded to keep this analysis on the MovieLens taxonomy.

In [2]:
queries = {
    "Q1_year_month_genre": ("""
WITH monthly AS (
    SELECT d.calendar_year AS year, d.month_number AS month,
           g.genre_name AS genre, COUNT(*) AS rating_count,
           AVG(r.rating) AS average_rating
    FROM fact_rating r
    JOIN dim_date d ON d.date_key = r.date_key
    JOIN bridge_movie_genre b ON b.movie_key = r.movie_key
    JOIN dim_genre g ON g.genre_key = b.genre_key AND g.source_name = 'MovieLens'
    GROUP BY d.calendar_year, d.month_number, g.genre_key
), yearly AS (
    SELECT d.calendar_year AS year, NULL AS month,
           g.genre_name AS genre, COUNT(*) AS rating_count,
           AVG(r.rating) AS average_rating
    FROM fact_rating r
    JOIN dim_date d ON d.date_key = r.date_key
    JOIN bridge_movie_genre b ON b.movie_key = r.movie_key
    JOIN dim_genre g ON g.genre_key = b.genre_key AND g.source_name = 'MovieLens'
    GROUP BY d.calendar_year, g.genre_key
)
SELECT 'month' AS period_grain, year, month, genre, rating_count, average_rating FROM monthly
UNION ALL
SELECT 'year' AS period_grain, year, month, genre, rating_count, average_rating FROM yearly
ORDER BY year, period_grain, month, genre
""", ()),
    "Q2_genre_by_user_activity_band": ("""
WITH user_activity AS (
    SELECT user_key, COUNT(*) AS lifetime_ratings
    FROM fact_rating
    GROUP BY user_key
), user_band AS (
    SELECT user_key,
           CASE WHEN lifetime_ratings <= 100 THEN 'low (<=100)'
                WHEN lifetime_ratings <= 500 THEN 'medium (101-500)'
                ELSE 'high (>500)' END AS activity_band
    FROM user_activity
), sliced AS (
    SELECT r.rating, b.genre_key, ub.activity_band
    FROM fact_rating r
    JOIN dim_date d ON d.date_key = r.date_key
    JOIN user_band ub ON ub.user_key = r.user_key
    JOIN bridge_movie_genre b ON b.movie_key = r.movie_key
    JOIN dim_genre g ON g.genre_key = b.genre_key AND g.source_name = 'MovieLens'
    WHERE d.calendar_date >= ? AND d.calendar_date < ?
)
SELECT g.genre_name AS genre,
       COUNT(*) AS all_rating_events,
       SUM(CASE WHEN s.activity_band = 'low (<=100)' THEN 1 ELSE 0 END) AS low_events,
       AVG(CASE WHEN s.activity_band = 'low (<=100)' THEN s.rating END) AS low_average,
       SUM(CASE WHEN s.activity_band = 'medium (101-500)' THEN 1 ELSE 0 END) AS medium_events,
       AVG(CASE WHEN s.activity_band = 'medium (101-500)' THEN s.rating END) AS medium_average,
       SUM(CASE WHEN s.activity_band = 'high (>500)' THEN 1 ELSE 0 END) AS high_events,
       AVG(CASE WHEN s.activity_band = 'high (>500)' THEN s.rating END) AS high_average
FROM sliced s
JOIN dim_genre g ON g.genre_key = s.genre_key
GROUP BY g.genre_key
ORDER BY all_rating_events DESC
""", ("2010-01-01", "2016-01-01")),
    "Q3_top_bottom_genres_by_year": ("""
WITH genre_year AS (
    SELECT d.calendar_year AS year, g.genre_key, g.genre_name AS genre,
           COUNT(*) AS rating_count, AVG(r.rating) AS average_rating
    FROM fact_rating r
    JOIN dim_date d ON d.date_key = r.date_key
    JOIN bridge_movie_genre b ON b.movie_key = r.movie_key
    JOIN dim_genre g ON g.genre_key = b.genre_key AND g.source_name = 'MovieLens'
    GROUP BY d.calendar_year, g.genre_key
    HAVING COUNT(*) >= 1000
), ranked AS (
    SELECT *,
           RANK() OVER (PARTITION BY year ORDER BY average_rating DESC) AS best_rank,
           RANK() OVER (PARTITION BY year ORDER BY average_rating ASC) AS lowest_rank
    FROM genre_year
)
SELECT year, 'highest' AS position, genre, rating_count, average_rating
FROM ranked WHERE best_rank = 1
UNION ALL
SELECT year, 'lowest' AS position, genre, rating_count, average_rating
FROM ranked WHERE lowest_rank = 1
ORDER BY year, position, genre
""", ()),
    "Q4_tag_applications_by_tag_genre_year": ("""
SELECT d.calendar_year AS year, t.tag_text AS user_tag, g.genre_name AS genre,
       COUNT(*) AS tag_applications
FROM fact_tag f
JOIN dim_date d ON d.date_key = f.date_key
JOIN dim_user_tag t ON t.user_tag_key = f.user_tag_key
JOIN bridge_movie_genre b ON b.movie_key = f.movie_key
JOIN dim_genre g ON g.genre_key = b.genre_key AND g.source_name = 'MovieLens'
GROUP BY d.calendar_year, t.user_tag_key, g.genre_key
ORDER BY tag_applications DESC, year, user_tag, genre
""", ()),
    "Q5_country_release_decade": ("""
WITH rating_by_movie AS (
    SELECT movie_key, COUNT(*) AS rating_count, SUM(rating) AS rating_sum
    FROM fact_rating
    GROUP BY movie_key
)
SELECT COALESCE(c.country_name, 'No Wikidata country') AS wikidata_country,
       CASE WHEN m.release_date IS NULL THEN 'Unknown release decade'
            ELSE CAST((CAST(substr(m.release_date, 1, 4) AS INTEGER) / 10) * 10 AS TEXT) || 's'
       END AS release_decade,
       COUNT(DISTINCT m.movie_key) AS movie_count,
       SUM(COALESCE(r.rating_count, 0)) AS rating_count,
       CASE WHEN SUM(COALESCE(r.rating_count, 0)) = 0 THEN NULL
            ELSE SUM(COALESCE(r.rating_sum, 0.0)) / SUM(r.rating_count)
       END AS average_rating
FROM dim_movie m
LEFT JOIN rating_by_movie r ON r.movie_key = m.movie_key
LEFT JOIN bridge_movie_country bc ON bc.movie_key = m.movie_key
LEFT JOIN dim_country c ON c.country_key = bc.country_key
WHERE m.is_current = 1
GROUP BY COALESCE(c.country_name, 'No Wikidata country'), release_decade
ORDER BY rating_count DESC, wikidata_country, release_decade
""", ()),
}

# Companion coverage figures make the country analysis denominator explicit.
coverage_sql = """
SELECT CASE WHEN m.wikidata_qid IS NULL THEN 'no exact Wikidata match (unmatched or ambiguous)'
            WHEN EXISTS (SELECT 1 FROM bridge_movie_country bc WHERE bc.movie_key = m.movie_key)
                 THEN 'matched with country'
            ELSE 'matched without country' END AS coverage_group,
       COUNT(DISTINCT m.movie_key) AS movies,
       COUNT(DISTINCT CASE WHEN r.movie_key IS NOT NULL THEN m.movie_key END) AS movies_with_ratings,
       SUM(COALESCE(r.rating_count, 0)) AS rating_events
FROM dim_movie m
LEFT JOIN (SELECT movie_key, COUNT(*) AS rating_count FROM fact_rating GROUP BY movie_key) r
       ON r.movie_key = m.movie_key
WHERE m.is_current = 1
GROUP BY coverage_group
ORDER BY coverage_group
"""
print(f"Defined {len(queries)} OLAP analysis queries.")

Defined 5 OLAP analysis queries.


## Run and time the five analyses

The result count is the full number of aggregate rows returned before preview truncation. Query 2's time window is 2010-01-01 through 2015-12-31 (UTC). Query 4 describes user-applied tag events, not movie views. Query 5 averages are event-weighted across ratings; a movie with multiple Wikidata countries contributes its rating events to each associated country, while the bridge does not multiply rows in the rating fact.

In [3]:
def run_timed(sql, params=()):
    started = time.perf_counter()
    cursor = con.execute(sql, params)
    rows = cursor.fetchall()
    elapsed = time.perf_counter() - started
    return rows, elapsed


def explain(sql, params=()):
    return [row["detail"] for row in con.execute("EXPLAIN QUERY PLAN " + sql, params)]


def preview(rows, limit=12):
    return [dict(row) for row in rows[:limit]]

results = {}
for name, (sql, params) in queries.items():
    rows, seconds = run_timed(sql, params)
    results[name] = {
        "elapsed_seconds": round(seconds, 6),
        "result_rows": len(rows),
        "preview": preview(rows),
        "query_plan": explain(sql, params),
    }
    print(f"{name}: {seconds:.3f} seconds; {len(rows):,} result rows")

coverage_rows, coverage_seconds = run_timed(coverage_sql)
results["Q5_coverage"] = {
    "elapsed_seconds": round(coverage_seconds, 6),
    "result_rows": len(coverage_rows),
    "preview": preview(coverage_rows, 20),
    "query_plan": explain(coverage_sql),
}
print(f"Q5_coverage: {coverage_seconds:.3f} seconds; {len(coverage_rows):,} result rows")

Q1_year_month_genre: 162.935 seconds; 4,792 result rows
Q2_genre_by_user_activity_band: 25.890 seconds; 20 result rows
Q3_top_bottom_genres_by_year: 81.855 seconds; 40 result rows
Q4_tag_applications_by_tag_genre_year: 2.268 seconds; 324,427 result rows
Q5_country_release_decade: 134.937 seconds; 590 result rows
Q5_coverage: 0.799 seconds; 3 result rows


In [4]:
# Compact, readable result previews and actual query plan/index choices.
from IPython.display import display, Markdown

display(Markdown("### Query result previews"))
for name, result in results.items():
    display(Markdown(f"**{name}** — {result['result_rows']:,} rows; {result['elapsed_seconds']:.3f} seconds"))
    display(result["preview"])

display(Markdown("### SQLite query plans"))
for name, result in results.items():
    print(f"\n{name}")
    for step in result["query_plan"]:
        print(" -", step)

### Query result previews
**Q1_year_month_genre** — 4,792 rows; 162.935 seconds
[{"period_grain": "month", "year": 1995, "month": 1, "genre": "Comedy", "rating_count": 2, "average_rating": 3.0}, {"period_grain": "month", "year": 1995, "month": 1, "genre": "Crime", "rating_count": 2, "average_rating": 3.0}, {"period_grain": "month", "year": 1995, "month": 1, "genre": "Drama", "rating_count": 1, "average_rating": 4.0}, {"period_grain": "month", "year": 1995, "month": 1, "genre": "Fantasy", "rating_count": 1, "average_rating": 4.0}, {"period_grain": "month", "year": 1995, "month": 1, "genre": "Mystery", "rating_count": 1, "average_rating": 5.0}, {"period_grain": "month", "year": 1995, "month": 1, "genre": "Romance", "rating_count": 1, "average_rating": 4.0}, {"period_grain": "month", "year": 1995, "month": 1, "genre": "Thriller", "rating_count": 2, "average_rating": 4.0}, {"period_grain": "year", "year": 1995, "month": null, "genre": "Comedy", "rating_count": 2, "average_rating": 3.0}, {"

In [5]:
run_record = {
    "metadata": metadata,
    "query_order": list(queries) + ["Q5_coverage"],
    "results": results,
}
OUTPUT_PATH.write_text(json.dumps(run_record, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(f"Saved machine-readable evidence to {OUTPUT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Database opened read-only; size {DB_BYTES:,} bytes.")

Saved machine-readable evidence to outputs/olap/olap_run.json
Database opened read-only; size 5,700,124,672 bytes.


## Interpretation notes and limitations

- Genre and country are multi-valued dimensions. Measures are interpreted per assigned genre/country, so totals across categories need not equal the overall rating-event total.
- User activity bands use the full MovieLens 20M history, so they are descriptive groups rather than future-safe prediction features.
- Wikidata is a current metadata snapshot. Release period and country should not be interpreted as metadata known to MovieLens users at rating time.
- Runtime figures describe this machine, database file, SQLite build, sequential query order, and cache state. They are reproducibility evidence, not a universal benchmark.